# Bizi Model Recovery - FIXED

This notebook rebuilds the Bizi recommender from the original Drive source files. The previous recovery notebook failed because `bizi_sponsored_post_events.csv` was a derived file and was not present on Drive. This version recreates that file first, then rebuilds the serving Random Forest pipeline.


In [1]:
!pip -q install sentence-transformers

import os, json, csv, zipfile, shutil, time
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

from google.colab import drive
drive.mount('/content/drive')

DATASET_DIR = '/content/drive/MyDrive/Influencer brand dataset'
POST_INFO = os.path.join(DATASET_DIR, 'post_info.txt')
JSON_ZIP = os.path.join(DATASET_DIR, 'json_files.zip')
CREATOR_ZIP = os.path.join(DATASET_DIR, 'profiles_influencers.zip')
BRAND_ZIP = os.path.join(DATASET_DIR, 'profiles_brands.zip')
EVENTS_CSV = os.path.join(DATASET_DIR, 'bizi_sponsored_post_events.csv')

for p in [POST_INFO, JSON_ZIP, CREATOR_ZIP, BRAND_ZIP]:
    print(os.path.basename(p), '->', os.path.exists(p))

assert all(os.path.exists(p) for p in [POST_INFO, JSON_ZIP, CREATOR_ZIP, BRAND_ZIP]), 'A source dataset file is missing.'
print('SOURCE DATA READY')


Mounted at /content/drive
post_info.txt -> True
json_files.zip -> True
profiles_influencers.zip -> True
profiles_brands.zip -> True
SOURCE DATA READY


In [2]:
import os
import json
import csv
import zipfile
import shutil
import time
from collections import Counter

DATASET_DIR = "/content/drive/MyDrive/Influencer brand dataset"

POST_INFO = os.path.join(DATASET_DIR, "post_info.txt")
JSON_ZIP = os.path.join(DATASET_DIR, "json_files.zip")

LOCAL_OUT = "/content/bizi_sponsored_post_events.csv"
DRIVE_OUT = os.path.join(DATASET_DIR, "bizi_sponsored_post_events.csv")

# ---------------------------------------------------------
# STEP 1: Collect sponsored post references
# ---------------------------------------------------------

print("Reading post_info.txt...")

sponsored_posts = []

with open(POST_INFO, "r", encoding="utf-8", errors="replace") as f:
    for line in f:
        line = line.rstrip("\n")

        parts = line.split("\t")

        if len(parts) < 4:
            continue

        post_id = parts[0]
        creator_username = parts[1]
        sponsorship_label = parts[2]
        json_filename = parts[3]

        if sponsorship_label == "1":
            sponsored_posts.append({
                "post_id": post_id,
                "creator_username": creator_username,
                "json_filename": json_filename
            })

print("Sponsored posts found:", len(sponsored_posts))


# ---------------------------------------------------------
# STEP 2: Read corresponding JSON files
# ---------------------------------------------------------

fieldnames = [
    "post_id",
    "creator_username",
    "creator_id",
    "sponsor_username",
    "sponsor_id",
    "timestamp",
    "datetime_utc",
    "likes",
    "comments",
    "caption",
    "json_filename"
]

relationships = []

missing_json = 0
json_errors = 0
sponsored_with_edges = 0
empty_edges = 0

sponsor_counter = Counter()
creator_counter = Counter()

start = time.time()

with zipfile.ZipFile(JSON_ZIP, "r") as z:

    total = len(sponsored_posts)

    for i, item in enumerate(sponsored_posts, start=1):

        json_path = "json/" + item["json_filename"]

        try:
            with z.open(json_path) as jf:
                data = json.load(jf)

        except KeyError:
            missing_json += 1
            continue

        except Exception:
            json_errors += 1
            continue

        # -------------------------------------------------
        # Creator information
        # -------------------------------------------------

        owner = data.get("owner") or {}

        creator_id = owner.get("id")

        # Prefer JSON username if available,
        # otherwise use post_info username.
        creator_username = (
            owner.get("username")
            or item["creator_username"]
        )

        # -------------------------------------------------
        # Post information
        # -------------------------------------------------

        timestamp = data.get("taken_at_timestamp")

        likes = (
            (data.get("edge_media_preview_like") or {})
            .get("count")
        )

        comments = (
            (data.get("edge_media_to_comment") or {})
            .get("count")
        )

        # Caption
        caption = ""

        caption_edges = (
            (data.get("edge_media_to_caption") or {})
            .get("edges")
            or []
        )

        if caption_edges:
            try:
                caption = (
                    caption_edges[0]
                    .get("node", {})
                    .get("text", "")
                )
            except Exception:
                caption = ""

        # -------------------------------------------------
        # Sponsor information
        # -------------------------------------------------

        sponsor_field = data.get("edge_media_to_sponsor_user")

        if not sponsor_field:
            empty_edges += 1
            continue

        edges = sponsor_field.get("edges") or []

        if not edges:
            empty_edges += 1
            continue

        sponsored_with_edges += 1

        for edge in edges:

            node = edge.get("node") or {}
            sponsor = node.get("sponsor") or {}

            sponsor_id = sponsor.get("id")
            sponsor_username = sponsor.get("username")

            # Ignore malformed sponsor edges
            if not sponsor_username and not sponsor_id:
                continue

            # Convert timestamp to UTC-readable form
            datetime_utc = ""

            if timestamp is not None:
                try:
                    import datetime as dt

                    datetime_utc = dt.datetime.fromtimestamp(
                        int(timestamp),
                        tz=dt.timezone.utc
                    ).isoformat()
                except Exception:
                    datetime_utc = ""

            row = {
                "post_id": item["post_id"],
                "creator_username": creator_username,
                "creator_id": creator_id,
                "sponsor_username": sponsor_username,
                "sponsor_id": sponsor_id,
                "timestamp": timestamp,
                "datetime_utc": datetime_utc,
                "likes": likes,
                "comments": comments,
                "caption": caption,
                "json_filename": item["json_filename"]
            }

            relationships.append(row)

            sponsor_counter[sponsor_username] += 1
            creator_counter[creator_username] += 1

        # Progress
        if i % 10000 == 0:
            elapsed = time.time() - start
            rate = i / elapsed if elapsed else 0

            print(
                f"{i:,}/{total:,} "
                f"({i/total:.1%}) | "
                f"relationships={len(relationships):,} | "
                f"{rate:.1f} posts/sec"
            )


# ---------------------------------------------------------
# STEP 3: Save locally FIRST
# ---------------------------------------------------------

print("\nWriting local CSV...")

with open(
    LOCAL_OUT,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=fieldnames
    )

    writer.writeheader()
    writer.writerows(relationships)


# ---------------------------------------------------------
# STEP 4: Attempt Drive copy
# ---------------------------------------------------------

drive_saved = False

try:
    shutil.copy2(LOCAL_OUT, DRIVE_OUT)
    drive_saved = True
    print("Drive copy successful:")
    print(DRIVE_OUT)

except Exception as e:
    print("Drive copy failed:")
    print(repr(e))
    print("Local file is still safe at:")
    print(LOCAL_OUT)


# ---------------------------------------------------------
# STEP 5: Final audit
# ---------------------------------------------------------

unique_sponsors = set()
unique_creators = set()
unique_pairs = set()

for r in relationships:

    sponsor = r["sponsor_username"]
    creator = r["creator_username"]

    unique_sponsors.add(sponsor)
    unique_creators.add(creator)
    unique_pairs.add((sponsor, creator))


print("\n" + "=" * 60)
print("FINAL SPONSORED-POST AUDIT")
print("=" * 60)

print("Sponsored posts examined:       ", len(sponsored_posts))
print("Posts with sponsor edges:       ", sponsored_with_edges)
print("Empty/missing sponsor edges:    ", empty_edges)
print("Missing JSON files:              ", missing_json)
print("JSON errors:                     ", json_errors)

print()
print("Relationship rows:               ", len(relationships))
print("Unique sponsors:                 ", len(unique_sponsors))
print("Unique creators:                 ", len(unique_creators))
print("Unique brand-creator pairs:      ", len(unique_pairs))

print()
print("Local CSV:", LOCAL_OUT)
print("Drive CSV exists:", os.path.exists(DRIVE_OUT))

print("\nExpected from our previous audit:")
print("  Sponsored posts:       221,710")
print("  Sponsor relationships:   8,655")
print("  Unique sponsors:         4,151")
print("  Unique creators:         2,362")
print("  Unique pairs:            6,507")

Reading post_info.txt...
Sponsored posts found: 221710
20,000/221,710 (9.0%) | relationships=673 | 153.0 posts/sec

Writing local CSV...
Drive copy failed:
OSError(30, 'Read-only file system')
Local file is still safe at:
/content/bizi_sponsored_post_events.csv

FINAL SPONSORED-POST AUDIT
Sponsored posts examined:        221710
Posts with sponsor edges:        8655
Empty/missing sponsor edges:     213055
Missing JSON files:               0
JSON errors:                      0

Relationship rows:                8655
Unique sponsors:                  4151
Unique creators:                  2362
Unique brand-creator pairs:       6507

Local CSV: /content/bizi_sponsored_post_events.csv
Drive CSV exists: False

Expected from our previous audit:
  Sponsored posts:       221,710
  Sponsor relationships:   8,655
  Unique sponsors:         4,151
  Unique creators:         2,362
  Unique pairs:            6,507


In [3]:
import os
import json
import zipfile
import time
import pandas as pd
import numpy as np
from collections import defaultdict

DATASET_DIR = "/content/drive/MyDrive/Influencer brand dataset"
POST_INFO = os.path.join(DATASET_DIR, "post_info.txt")
JSON_ZIP = os.path.join(DATASET_DIR, "json_files.zip")

EVENTS_CSV = "/content/bizi_sponsored_post_events.csv"

# ---------------------------------------------------------
# 1. Load the sponsored campaign events we just created
# ---------------------------------------------------------

events = pd.read_csv(EVENTS_CSV)

events["timestamp"] = pd.to_numeric(
    events["timestamp"],
    errors="coerce"
)

events["likes"] = pd.to_numeric(
    events["likes"],
    errors="coerce"
)

events["comments"] = pd.to_numeric(
    events["comments"],
    errors="coerce"
)

events["campaign_datetime"] = pd.to_datetime(
    events["timestamp"],
    unit="s",
    utc=True,
    errors="coerce"
)

print("Sponsored event rows:", len(events))
print("Unique creators:", events["creator_username"].nunique())

# ---------------------------------------------------------
# 2. Get the creators involved in real sponsorships
# ---------------------------------------------------------

target_creators = set(
    events["creator_username"]
    .dropna()
    .astype(str)
)

print("Target creators:", len(target_creators))

# ---------------------------------------------------------
# 3. Read post_info.txt and collect posts belonging
#    to those creators.
#
#    We do NOT need to inspect JSON yet.
# ---------------------------------------------------------

creator_posts = defaultdict(list)

total_indexed = 0
relevant_indexed = 0

print("\nIndexing post_info.txt...")

with open(
    POST_INFO,
    "r",
    encoding="utf-8",
    errors="replace"
) as f:

    for line in f:

        line = line.rstrip("\n")

        parts = line.split("\t")

        if len(parts) < 4:
            continue

        post_id = parts[0]
        username = parts[1]
        sponsorship = parts[2]
        json_filename = parts[3]

        total_indexed += 1

        if username not in target_creators:
            continue

        relevant_indexed += 1

        creator_posts[username].append({
            "post_id": post_id,
            "sponsorship": int(sponsorship),
            "json_filename": json_filename
        })

print("Total posts indexed:", total_indexed)
print("Relevant creator posts:", relevant_indexed)

# ---------------------------------------------------------
# 4. Load JSON metadata for those creators
# ---------------------------------------------------------

organic_posts = []

json_errors = 0
missing_json = 0

start = time.time()

with zipfile.ZipFile(JSON_ZIP, "r") as z:

    processed = 0

    for creator, posts in creator_posts.items():

        for p in posts:

            json_path = "json/" + p["json_filename"]

            try:
                with z.open(json_path) as jf:
                    data = json.load(jf)

            except KeyError:
                missing_json += 1
                continue

            except Exception:
                json_errors += 1
                continue

            timestamp = data.get("taken_at_timestamp")

            likes = (
                (data.get("edge_media_preview_like") or {})
                .get("count")
            )

            comments = (
                (data.get("edge_media_to_comment") or {})
                .get("count")
            )

            # We only need organic posts for baseline.
            if p["sponsorship"] == 0:

                organic_posts.append({
                    "post_id": p["post_id"],
                    "creator_username": creator,
                    "timestamp": timestamp,
                    "likes": likes,
                    "comments": comments
                })

            processed += 1

        if len(creator_posts) and processed % 50000 == 0:
            elapsed = time.time() - start
            print(
                f"Processed {processed:,} relevant posts "
                f"| organic={len(organic_posts):,}"
            )

print("\nJSON processing complete.")

print("Organic posts collected:", len(organic_posts))
print("Missing JSON:", missing_json)
print("JSON errors:", json_errors)

# ---------------------------------------------------------
# 5. Convert to DataFrame
# ---------------------------------------------------------

organic_df = pd.DataFrame(organic_posts)

organic_df["timestamp"] = pd.to_numeric(
    organic_df["timestamp"],
    errors="coerce"
)

organic_df["likes"] = pd.to_numeric(
    organic_df["likes"],
    errors="coerce"
)

organic_df["comments"] = pd.to_numeric(
    organic_df["comments"],
    errors="coerce"
)

organic_df["datetime"] = pd.to_datetime(
    organic_df["timestamp"],
    unit="s",
    utc=True,
    errors="coerce"
)

organic_df = organic_df.dropna(
    subset=["creator_username", "timestamp"]
)

organic_df = organic_df.sort_values(
    ["creator_username", "timestamp"]
)

print("\nOrganic dataframe:", organic_df.shape)

# ---------------------------------------------------------
# 6. Load creator follower counts
# ---------------------------------------------------------

CREATOR_ZIP = os.path.join(
    DATASET_DIR,
    "profiles_influencers.zip"
)

creator_followers = {}

print("\nLoading creator follower counts...")

with zipfile.ZipFile(CREATOR_ZIP, "r") as z:

    names = [
        n for n in z.namelist()
        if n.startswith("users_influencers_SPOD/")
        and not n.endswith("/")
    ]

    for name in names:

        username = name.split("/")[-1]

        try:
            with z.open(name) as f:
                raw = f.read().decode(
                    "utf-8",
                    errors="replace"
                ).rstrip("\n")

            fields = raw.split("\t")

            if len(fields) < 2:
                continue

            followers = pd.to_numeric(
                fields[1],
                errors="coerce"
            )

            if pd.notna(followers):
                creator_followers[username] = float(
                    followers
                )

        except Exception:
            continue

print(
    "Creators with follower counts:",
    len(creator_followers)
)

# ---------------------------------------------------------
# 7. Add follower count
# ---------------------------------------------------------

organic_df["followers"] = organic_df[
    "creator_username"
].map(creator_followers)

# Calculate engagement rate only when follower
# count is positive and engagement values exist.

organic_df["engagement_rate"] = np.where(
    (
        organic_df["followers"].notna()
        & (organic_df["followers"] > 0)
        & organic_df["likes"].notna()
        & organic_df["comments"].notna()
    ),
    (
        (
            organic_df["likes"]
            + organic_df["comments"]
        )
        / organic_df["followers"]
        * 100
    ),
    np.nan
)

# ---------------------------------------------------------
# 8. Build chronological lookup per creator
# ---------------------------------------------------------

creator_history = {}

for creator, group in organic_df.groupby(
    "creator_username"
):

    group = group.sort_values("timestamp")

    creator_history[creator] = {
        "timestamp": group["timestamp"].to_numpy(),
        "engagement_rate": group[
            "engagement_rate"
        ].to_numpy(),
        "post_id": group["post_id"].to_numpy()
    }

print(
    "Creators with organic history:",
    len(creator_history)
)

# ---------------------------------------------------------
# 9. Calculate pre-campaign baseline
# ---------------------------------------------------------

baseline_rows = []

for idx, event in events.iterrows():

    creator = event["creator_username"]

    campaign_timestamp = event["timestamp"]

    history = creator_history.get(creator)

    if history is None:
        baseline_rows.append({
            "baseline_post_count": 0,
            "baseline_er_mean": np.nan,
            "baseline_er_median": np.nan,
            "baseline_last_post_timestamp": np.nan,
            "baseline_first_post_timestamp": np.nan
        })
        continue

    timestamps = history["timestamp"]

    # STRICTLY before campaign
    mask = timestamps < campaign_timestamp

    positions = np.where(mask)[0]

    if len(positions) == 0:
        baseline_rows.append({
            "baseline_post_count": 0,
            "baseline_er_mean": np.nan,
            "baseline_er_median": np.nan,
            "baseline_last_post_timestamp": np.nan,
            "baseline_first_post_timestamp": np.nan
        })
        continue

    # Take the latest 10 organic posts
    positions = positions[-10:]

    er_values = history["engagement_rate"][positions]

    er_values = er_values[
        np.isfinite(er_values)
    ]

    if len(er_values) == 0:
        baseline_mean = np.nan
        baseline_median = np.nan
    else:
        baseline_mean = float(
            np.mean(er_values)
        )

        baseline_median = float(
            np.median(er_values)
        )

    baseline_timestamps = timestamps[positions]

    baseline_rows.append({
        "baseline_post_count": len(positions),
        "baseline_er_mean": baseline_mean,
        "baseline_er_median": baseline_median,
        "baseline_last_post_timestamp": int(
            baseline_timestamps[-1]
        ),
        "baseline_first_post_timestamp": int(
            baseline_timestamps[0]
        )
    })

# ---------------------------------------------------------
# 10. Merge baseline with campaign events
# ---------------------------------------------------------

baseline_df = pd.DataFrame(baseline_rows)

campaign_df = pd.concat(
    [
        events.reset_index(drop=True),
        baseline_df
    ],
    axis=1
)

# ---------------------------------------------------------
# 11. Campaign engagement rate
# ---------------------------------------------------------

campaign_df["creator_followers"] = campaign_df[
    "creator_username"
].map(creator_followers)

campaign_df["campaign_er"] = np.where(
    (
        campaign_df["creator_followers"].notna()
        & (campaign_df["creator_followers"] > 0)
        & campaign_df["likes"].notna()
        & campaign_df["comments"].notna()
    ),
    (
        (
            campaign_df["likes"]
            + campaign_df["comments"]
        )
        / campaign_df["creator_followers"]
        * 100
    ),
    np.nan
)

# ---------------------------------------------------------
# 12. Engagement uplift
# ---------------------------------------------------------

campaign_df["engagement_uplift"] = np.where(
    (
        campaign_df["baseline_er_median"].notna()
        & (campaign_df["baseline_er_median"] > 0)
        & campaign_df["campaign_er"].notna()
    ),
    (
        campaign_df["campaign_er"]
        / campaign_df["baseline_er_median"]
        - 1
    ),
    np.nan
)

# ---------------------------------------------------------
# 13. Save locally
# ---------------------------------------------------------

OUT = "/content/bizi_campaign_engagement_events.csv"

campaign_df.to_csv(
    OUT,
    index=False
)

# ---------------------------------------------------------
# 14. Summary
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("CAMPAIGN ENGAGEMENT DATASET")
print("=" * 60)

print("Sponsored events:", len(campaign_df))

print(
    "Events with baseline:",
    campaign_df["baseline_er_median"].notna().sum()
)

print(
    "Events with campaign ER:",
    campaign_df["campaign_er"].notna().sum()
)

print(
    "Events with usable uplift:",
    campaign_df["engagement_uplift"].notna().sum()
)

print(
    "Median baseline ER:",
    campaign_df["baseline_er_median"].median()
)

print(
    "Median campaign ER:",
    campaign_df["campaign_er"].median()
)

print(
    "Median engagement uplift:",
    campaign_df["engagement_uplift"].median()
)

print("\nSaved locally:")
print(OUT)

Sponsored event rows: 8655
Unique creators: 2362
Target creators: 2362

Indexing post_info.txt...
Total posts indexed: 1601074
Relevant creator posts: 140696

JSON processing complete.
Organic posts collected: 114582
Missing JSON: 0
JSON errors: 0

Organic dataframe: (114582, 6)

Loading creator follower counts...
Creators with follower counts: 38113
Creators with organic history: 2344

CAMPAIGN ENGAGEMENT DATASET
Sponsored events: 8655
Events with baseline: 8397
Events with campaign ER: 8655
Events with usable uplift: 8397
Median baseline ER: 2.3908418599939214
Median campaign ER: 2.038870727279489
Median engagement uplift: -0.11003460207612448

Saved locally:
/content/bizi_campaign_engagement_events.csv


In [4]:
import os
import zipfile
import numpy as np
import pandas as pd

DATASET_DIR = "/content/drive/MyDrive/Influencer brand dataset"

EVENTS_CSV = "/content/bizi_sponsored_post_events.csv"
CAMPAIGN_CSV = "/content/bizi_campaign_engagement_events.csv"

# ---------------------------------------------------------
# 1. Load sponsored relationship events
# ---------------------------------------------------------

events = pd.read_csv(EVENTS_CSV)

events["timestamp"] = pd.to_numeric(
    events["timestamp"],
    errors="coerce"
)

events["creator_username"] = (
    events["creator_username"].astype(str)
)

events["sponsor_username"] = (
    events["sponsor_username"].astype(str)
)

print("Sponsored event rows:", len(events))

# ---------------------------------------------------------
# 2. Build unique brand-creator relationships
#
# IMPORTANT:
# One pair is one positive observation.
# Collaboration count is NOT used as a predictor.
# ---------------------------------------------------------

pair_df = (
    events
    .groupby(
        ["sponsor_username", "creator_username"],
        as_index=False
    )
    .agg(
        collaboration_count=("post_id", "count"),
        first_collaboration=("timestamp", "min"),
        last_collaboration=("timestamp", "max")
    )
)

pair_df = pair_df.rename(
    columns={
        "sponsor_username": "brand_username"
    }
)

print("Unique brand-creator pairs:", len(pair_df))

# ---------------------------------------------------------
# 3. Load profiles only for relevant brands/creators
# ---------------------------------------------------------

target_brands = set(
    pair_df["brand_username"]
)

target_creators = set(
    pair_df["creator_username"]
)

print("Target brands:", len(target_brands))
print("Target creators:", len(target_creators))


def load_profiles(zip_path, prefix, targets):
    rows = []

    with zipfile.ZipFile(zip_path, "r") as z:

        names = [
            n for n in z.namelist()
            if n.startswith(prefix)
            and not n.endswith("/")
        ]

        for name in names:

            username = name.split("/")[-1]

            if username not in targets:
                continue

            try:
                with z.open(name) as f:
                    raw = f.read().decode(
                        "utf-8",
                        errors="replace"
                    ).rstrip("\n")

                fields = raw.split("\t")

                if len(fields) != 11:
                    continue

                rows.append({
                    "username": username,
                    "display_name": fields[0],
                    "followers": fields[1],
                    "following": fields[2],
                    "post_count": fields[3],
                    "external_website": fields[4],
                    # fields[5] intentionally ignored
                    "category": fields[6],
                    "bio": fields[7]
                    # email/phone/image intentionally ignored
                })

            except Exception:
                continue

    return pd.DataFrame(rows)


creator_profiles = load_profiles(
    os.path.join(
        DATASET_DIR,
        "profiles_influencers.zip"
    ),
    "users_influencers_SPOD/",
    target_creators
)

brand_profiles = load_profiles(
    os.path.join(
        DATASET_DIR,
        "profiles_brands.zip"
    ),
    "users_brands_SPOD/",
    target_brands
)

print("\nCreator profiles loaded:", len(creator_profiles))
print("Brand profiles loaded:", len(brand_profiles))

# Numeric conversion
for df in [creator_profiles, brand_profiles]:

    for col in [
        "followers",
        "following",
        "post_count"
    ]:

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )


# ---------------------------------------------------------
# 4. Join profiles to positive pairs
# ---------------------------------------------------------

brand_profiles = brand_profiles.rename(
    columns={
        "username": "brand_username",
        "display_name": "brand_display_name",
        "followers": "brand_followers",
        "following": "brand_following",
        "post_count": "brand_post_count",
        "external_website": "brand_website",
        "category": "brand_category",
        "bio": "brand_bio"
    }
)

creator_profiles = creator_profiles.rename(
    columns={
        "username": "creator_username",
        "display_name": "creator_display_name",
        "followers": "creator_followers",
        "following": "creator_following",
        "post_count": "creator_post_count",
        "external_website": "creator_website",
        "category": "creator_category",
        "bio": "creator_bio"
    }
)

positive_pairs = (
    pair_df
    .merge(
        brand_profiles,
        on="brand_username",
        how="inner"
    )
    .merge(
        creator_profiles,
        on="creator_username",
        how="inner"
    )
)

print(
    "\nProfile-complete positive pairs:",
    len(positive_pairs)
)

print(
    "Unique brands:",
    positive_pairs["brand_username"].nunique()
)

print(
    "Unique creators:",
    positive_pairs["creator_username"].nunique()
)

# Expected:
# 4220
# 2151 brands
# 1865 creators


# ---------------------------------------------------------
# 5. Create the exact text representation used previously
#
# display_name + category + bio
# ---------------------------------------------------------

def make_text(row, prefix):
    values = [
        row.get(f"{prefix}_display_name"),
        row.get(f"{prefix}_category"),
        row.get(f"{prefix}_bio")
    ]

    values = [
        str(v).strip().lower()
        for v in values
        if pd.notna(v)
        and str(v).strip().lower() != "nan"
        and str(v).strip()
    ]

    return " ".join(values)


positive_pairs["brand_text"] = positive_pairs.apply(
    lambda r: make_text(r, "brand"),
    axis=1
)

positive_pairs["creator_text"] = positive_pairs.apply(
    lambda r: make_text(r, "creator"),
    axis=1
)


# ---------------------------------------------------------
# 6. Make sure organic history from previous step exists
# ---------------------------------------------------------

if "organic_df" not in globals():

    print(
        "\nWARNING: organic_df is not currently in memory."
    )

    print(
        "You need to rerun the previous organic-post "
        "baseline extraction cell before continuing."
    )

    raise RuntimeError(
        "organic_df is missing. Do not continue until "
        "the organic baseline dataset is loaded."
    )

print(
    "\nOrganic posts available:",
    len(organic_df)
)


# ---------------------------------------------------------
# 7. Build chronological organic history
# ---------------------------------------------------------

organic_work = organic_df.copy()

organic_work["timestamp"] = pd.to_numeric(
    organic_work["timestamp"],
    errors="coerce"
)

organic_work["engagement_rate"] = pd.to_numeric(
    organic_work["engagement_rate"],
    errors="coerce"
)

organic_work = organic_work.dropna(
    subset=[
        "creator_username",
        "timestamp"
    ]
)

organic_work = organic_work.sort_values(
    ["creator_username", "timestamp"]
)

creator_history = {}

for creator, group in organic_work.groupby(
    "creator_username"
):

    creator_history[creator] = {
        "timestamp": group["timestamp"].to_numpy(
            dtype=np.int64
        ),
        "er": group["engagement_rate"].to_numpy(
            dtype=float
        )
    }

print(
    "Creators with organic history:",
    len(creator_history)
)


# ---------------------------------------------------------
# 8. Function for temporal pre-campaign ER
#
# Uses ONLY the latest 10 organic posts STRICTLY before
# the cutoff timestamp.
# ---------------------------------------------------------

def temporal_baseline_er(
    creator,
    cutoff_timestamp,
    n_posts=10
):

    history = creator_history.get(creator)

    if history is None:
        return np.nan

    timestamps = history["timestamp"]
    ers = history["er"]

    # STRICTLY before campaign/cutoff
    pos = np.searchsorted(
        timestamps,
        cutoff_timestamp,
        side="left"
    )

    if pos == 0:
        return np.nan

    start = max(0, pos - n_posts)

    values = ers[start:pos]

    values = values[
        np.isfinite(values)
    ]

    if len(values) == 0:
        return np.nan

    return float(
        np.median(values)
    )


# ---------------------------------------------------------
# 9. Baseline ER for each positive pair
#
# Use FIRST collaboration only.
#
# This is critical:
# we don't allow later campaigns to influence the
# recommendation feature for that relationship.
# ---------------------------------------------------------

print(
    "\nCalculating first-collaboration baseline ER..."
)

positive_pairs["baseline_er_median"] = [
    temporal_baseline_er(
        creator,
        timestamp
    )
    for creator, timestamp
    in zip(
        positive_pairs["creator_username"],
        positive_pairs["first_collaboration"]
    )
]

positive_pairs["baseline_er_missing"] = (
    positive_pairs["baseline_er_median"]
    .isna()
    .astype(int)
)

print(
    "Positive pairs with baseline:",
    positive_pairs["baseline_er_median"].notna().sum()
)

print(
    "Positive pairs without baseline:",
    positive_pairs["baseline_er_median"].isna().sum()
)

print(
    "\nMedian first-collaboration baseline ER:",
    positive_pairs["baseline_er_median"].median()
)

print("\nDataset ready.")

Sponsored event rows: 8655
Unique brand-creator pairs: 6507
Target brands: 4151
Target creators: 2362

Creator profiles loaded: 2362
Brand profiles loaded: 2201

Profile-complete positive pairs: 4323
Unique brands: 2201
Unique creators: 1902

Organic posts available: 114582
Creators with organic history: 2344

Calculating first-collaboration baseline ER...
Positive pairs with baseline: 4210
Positive pairs without baseline: 113

Median first-collaboration baseline ER: 2.4259121435632136

Dataset ready.


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sentence_transformers import SentenceTransformer
import joblib, json, os

SEED = 42
NEGATIVES_PER_POSITIVE = 3

# Fixed brand-level split used by the final Bizi experiment
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(splitter.split(positive_pairs, groups=positive_pairs['brand_username']))

train_positive = positive_pairs.iloc[train_idx].copy()
test_positive = positive_pairs.iloc[test_idx].copy()

print('Train positive pairs:', len(train_positive))
print('Test positive pairs :', len(test_positive))
print('Train brands:', train_positive['brand_username'].nunique())
print('Test brands :', test_positive['brand_username'].nunique())

all_creators = np.array(positive_pairs['creator_username'].unique())
observed_by_brand = positive_pairs.groupby('brand_username')['creator_username'].apply(set).to_dict()

def add_profile_features(df):
    df = df.copy()
    df['log_creator_followers'] = np.log1p(pd.to_numeric(df['creator_followers'], errors='coerce').fillna(0))
    df['log_creator_following'] = np.log1p(pd.to_numeric(df['creator_following'], errors='coerce').fillna(0))
    df['log_creator_post_count'] = np.log1p(pd.to_numeric(df['creator_post_count'], errors='coerce').fillna(0))
    return df

creator_lookup = positive_pairs.drop_duplicates('creator_username').set_index('creator_username')

def make_candidates(positive_subset, seed):
    rng = np.random.default_rng(seed)
    positives = positive_subset.copy()
    positives['label'] = 1
    negative_rows = []

    for brand, group in positives.groupby('brand_username'):
        observed = observed_by_brand.get(brand, set())
        eligible = np.array([c for c in all_creators if c not in observed])
        n_needed = len(group) * NEGATIVES_PER_POSITIVE
        if len(eligible) < n_needed:
            raise RuntimeError(f'Not enough negative creators for {brand}')
        sampled = rng.choice(eligible, size=n_needed, replace=False)
        cutoff_dates = np.repeat(group['first_collaboration'].to_numpy(), NEGATIVES_PER_POSITIVE)
        first = group.iloc[0]

        for creator, cutoff in zip(sampled, cutoff_dates):
            cp = creator_lookup.loc[creator]
            negative_rows.append({
                'brand_username': brand, 'creator_username': creator,
                'brand_text': first['brand_text'],
                'brand_display_name': first['brand_display_name'],
                'brand_category': first['brand_category'], 'brand_bio': first['brand_bio'],
                'brand_followers': first['brand_followers'], 'brand_following': first['brand_following'],
                'brand_post_count': first['brand_post_count'],
                'creator_display_name': cp['creator_display_name'],
                'creator_category': cp['creator_category'], 'creator_bio': cp['creator_bio'],
                'creator_followers': cp['creator_followers'], 'creator_following': cp['creator_following'],
                'creator_post_count': cp['creator_post_count'], 'creator_text': cp['creator_text'],
                'first_collaboration': np.nan, 'feature_cutoff': cutoff, 'label': 0
            })

    negatives = pd.DataFrame(negative_rows)
    positives['feature_cutoff'] = positives['first_collaboration']
    return pd.concat([positives, negatives], ignore_index=True)

def add_temporal_engagement(candidates):
    candidates = candidates.copy()
    candidates['baseline_er_median'] = [
        temporal_baseline_er(c, cutoff)
        for c, cutoff in zip(candidates['creator_username'], candidates['feature_cutoff'])
    ]
    candidates['baseline_er_missing'] = candidates['baseline_er_median'].isna().astype(int)
    return candidates

train = add_temporal_engagement(add_profile_features(make_candidates(train_positive, SEED)))
test = add_temporal_engagement(add_profile_features(make_candidates(test_positive, SEED + 100000)))

print('Train candidates:', train.shape)
print('Test candidates :', test.shape)
print('Train labels:', train['label'].value_counts().to_dict())
print('Test labels :', test['label'].value_counts().to_dict())

# TF-IDF: fit on training text only
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=2, max_df=0.95, max_features=30000)
tfidf.fit(pd.concat([train['brand_text'], train['creator_text']]).fillna(''))

train_brand_vec = tfidf.transform(train['brand_text'].fillna(''))
train_creator_vec = tfidf.transform(train['creator_text'].fillna(''))
test_brand_vec = tfidf.transform(test['brand_text'].fillna(''))
test_creator_vec = tfidf.transform(test['creator_text'].fillna(''))

train['text_similarity'] = np.asarray(train_brand_vec.multiply(train_creator_vec).sum(axis=1)).ravel()
test['text_similarity'] = np.asarray(test_brand_vec.multiply(test_creator_vec).sum(axis=1)).ravel()

print('Loading MiniLM...')
semantic_model = SentenceTransformer('all-MiniLM-L6-v2')
print('MiniLM loaded.')

all_texts = pd.unique(pd.concat([train['brand_text'], train['creator_text'], test['brand_text'], test['creator_text']]).fillna(''))
embeddings = semantic_model.encode(all_texts.tolist(), normalize_embeddings=True, show_progress_bar=True)
embedding_lookup = dict(zip(all_texts, embeddings))

def semantic_scores(df):
    b = np.vstack([embedding_lookup[x] for x in df['brand_text'].fillna('')])
    c = np.vstack([embedding_lookup[x] for x in df['creator_text'].fillna('')])
    return np.sum(b*c, axis=1)

train['semantic_similarity'] = semantic_scores(train)
test['semantic_similarity'] = semantic_scores(test)

final_features = [
    'text_similarity', 'semantic_similarity',
    'log_creator_followers', 'log_creator_following',
    'log_creator_post_count', 'baseline_er_missing'
]

X_train = train[final_features].copy()
X_test = test[final_features].copy()

final_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('classifier', RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        max_features='sqrt',
        min_samples_leaf=5,
        random_state=SEED,
        n_jobs=-1,
    )),
])
final_model.fit(X_train, train['label'])

test_prob = final_model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(test['label'], test_prob)

def recall_at_k(df, scores, k):
    temp = df[['brand_username','label']].copy()
    temp['score'] = scores
    vals = []
    for brand, g in temp.groupby('brand_username'):
        positives = int(g['label'].sum())
        if positives == 0: continue
        top = g.sort_values('score', ascending=False).head(k)
        vals.append(int(top['label'].sum()) / positives)
    return float(np.mean(vals))

recall_1 = recall_at_k(test, test_prob, 1)
recall_3 = recall_at_k(test, test_prob, 3)
recall_5 = recall_at_k(test, test_prob, 5)

print('\n===== FINAL RANDOM FOREST MODEL =====')
print('ROC-AUC:', round(auc, 4))
print('Recall@1:', round(recall_1, 4))
print('Recall@3:', round(recall_3, 4))
print('Recall@5:', round(recall_5, 4))

# Save immediately to Drive
EXPORT_DIR = os.path.join(DATASET_DIR, 'Bizi_model_export')
os.makedirs(EXPORT_DIR, exist_ok=True)
joblib.dump(final_model, os.path.join(EXPORT_DIR, 'final_model.joblib'))
joblib.dump(tfidf, os.path.join(EXPORT_DIR, 'tfidf_vectorizer.joblib'))
train.to_csv(os.path.join(EXPORT_DIR, 'train_candidates.csv'), index=False)
test.to_csv(os.path.join(EXPORT_DIR, 'test_candidates.csv'), index=False)
creator_profiles.to_csv(os.path.join(EXPORT_DIR, 'creator_profiles.csv'), index=False)
brand_profiles.to_csv(os.path.join(EXPORT_DIR, 'brand_profiles.csv'), index=False)

with open(os.path.join(EXPORT_DIR, 'feature_config.json'), 'w') as f:
    json.dump({
        'features': final_features,
        'negative_sampling': 3,
        'split': 'GroupShuffleSplit test_size=0.20 random_state=42',
        'model': 'RandomForestClassifier',
        'semantic_model': 'all-MiniLM-L6-v2',
        'validation_selection_metric': 'Recall@3',
        'secondary_metric': 'ROC-AUC',
        'test_metrics': {
            'roc_auc': float(auc),
            'recall_at_1': float(recall_1),
            'recall_at_3': float(recall_3),
            'recall_at_5': float(recall_5),
        },
    }, f, indent=2)

print('\nMODEL SAVED TO:')
print(EXPORT_DIR)
print(os.listdir(EXPORT_DIR))


In [ ]:
# SAVE THE SUCCESSFULLY TRAINED MODEL LOCALLY FIRST
import os
import json
import joblib
import shutil

LOCAL_EXPORT = "/content/Bizi_model_export"
os.makedirs(LOCAL_EXPORT, exist_ok=True)

# Core model artifacts
joblib.dump(final_model, f"{LOCAL_EXPORT}/final_model.joblib")
joblib.dump(tfidf, f"{LOCAL_EXPORT}/tfidf_vectorizer.joblib")

# Data needed for the recommender
creator_profiles.to_csv(
    f"{LOCAL_EXPORT}/creator_profiles.csv",
    index=False
)

brand_profiles.to_csv(
    f"{LOCAL_EXPORT}/brand_profiles.csv",
    index=False
)

# Configuration
with open(f"{LOCAL_EXPORT}/feature_config.json", "w") as f:
    json.dump({
        "features": final_features,
        "negative_sampling": 3,
        "split": "GroupShuffleSplit test_size=0.20 random_state=42",
        "model": "RandomForestClassifier",
        "semantic_model": "all-MiniLM-L6-v2",
        "validation_selection_metric": "Recall@3",
        "secondary_metric": "ROC-AUC",
        "test_metrics": {
            "roc_auc": float(auc),
            "recall_at_1": float(recall_1),
            "recall_at_3": float(recall_3),
            "recall_at_5": float(recall_5)
        }
    }, f, indent=2)

# Save the MiniLM model too
semantic_model.save(f"{LOCAL_EXPORT}/all-MiniLM-L6-v2")

print("MODEL SAVED LOCALLY")
print(os.listdir(LOCAL_EXPORT))

In [7]:
from google.colab import drive
import os
import shutil

drive.mount("/content/drive", force_remount=True)

DRIVE_EXPORT = "/content/drive/MyDrive/Bizi_model_export"

os.makedirs(DRIVE_EXPORT, exist_ok=True)

shutil.copytree(
    "/content/Bizi_model_export",
    DRIVE_EXPORT,
    dirs_exist_ok=True
)

print("SAVED TO DRIVE:")
print(DRIVE_EXPORT)
print(os.listdir(DRIVE_EXPORT))

Mounted at /content/drive
SAVED TO DRIVE:
/content/drive/MyDrive/Bizi_model_export
['brand_profiles.csv', 'final_model.joblib', 'tfidf_vectorizer.joblib', 'feature_config.json', 'creator_profiles.csv', 'scaler.joblib', 'all-MiniLM-L6-v2']
